In [ ]:

from math import hypot



FLOOR = 0
WALL = 1

# 0 = lantai, 1 = tembok
GRID = [
    [1, 1, 1, 1, 1, 1, 1, 1, 1],
    [1, 0, 0, 0, 0, 0, 0, 0, 1],
    [1, 0, 1, 1, 0, 1, 1, 0, 1],
    [1, 0, 0, 0, 0, 0, 0, 0, 1],
    [1, 0, 1, 0, 1, 0, 1, 0, 1],
    [1, 0, 0, 0, 0, 0, 0, 0, 1],
    [1, 1, 1, 1, 1, 1, 1, 1, 1],
]

# Format posisi: (baris, kolom)
PLAYER_POS = (1, 5)
ENEMY_POS = (1, 1)
TARGET_POS = (5, 7)

SIGHT_RADIUS = 6




def in_bounds(pos):
    """True jika posisi berada di dalam grid."""
    row, col = pos

    return (
        0 <= row < len(GRID)
        and 0 <= col < len(GRID[row])
    )


def is_walkable(pos):
    """True jika posisi valid dan bukan tembok."""
    if not in_bounds(pos):
        return False

    row, col = pos
    return GRID[row][col] == FLOOR


def neighbors_4(pos):
    """Mengembalikan tetangga valid dalam empat arah."""
    row, col = pos

    candidates = [
        (row - 1, col),  # Atas
        (row + 1, col),  # Bawah
        (row, col - 1),  # Kiri
        (row, col + 1),  # Kanan
    ]

    return [
        cell for cell in candidates
        if is_walkable(cell)
    ]




def euclidean_distance(pos1, pos2):
    """Menghitung jarak lurus antara dua posisi grid."""
    return hypot(
        pos1[0] - pos2[0],
        pos1[1] - pos2[1]
    )



def bresenham_line(start, end):
    """
    Mengembalikan urutan cell yang dilewati garis
    dari start ke end, termasuk kedua titik.
    """
    row0, col0 = start
    row1, col1 = end

    # Bresenham menggunakan koordinat x dan y
    x0, y0 = col0, row0
    x1, y1 = col1, row1

    dx = abs(x1 - x0)
    sx = 1 if x0 < x1 else -1

    dy = -abs(y1 - y0)
    sy = 1 if y0 < y1 else -1

    error = dx + dy
    cells = []

    while True:
        cells.append((y0, x0))

        if x0 == x1 and y0 == y1:
            break

        double_error = 2 * error

        if double_error >= dy:
            error += dy
            x0 += sx

        if double_error <= dx:
            error += dx
            y0 += sy

    return cells




def can_enemy_see_player(
    enemy_pos,
    player_pos,
    radius
):
    """
    True jika player berada dalam radius
    dan garis pandang tidak terhalang tembok.
    """

    # Pastikan kedua posisi valid
    if not is_walkable(enemy_pos):
        return False

    if not is_walkable(player_pos):
        return False

    # Tahap pertama: cek radius Euclidean
    distance = euclidean_distance(
        enemy_pos,
        player_pos
    )

    if distance > radius:
        return False

    # Tahap kedua: cek tembok menggunakan Bresenham
    line = bresenham_line(enemy_pos, player_pos)

    # Titik awal dan akhir tidak dihitung sebagai penghalang
    for cell in line[1:-1]:
        if not is_walkable(cell):
            return False

    return True




if __name__ == "__main__":
    print("Enemy valid:", is_walkable(ENEMY_POS))
    print("Player valid:", is_walkable(PLAYER_POS))
    print("Target valid:", is_walkable(TARGET_POS))

    # Test 1: Player terlihat dalam radius
    assert can_enemy_see_player(
        ENEMY_POS, PLAYER_POS, SIGHT_RADIUS
    ) is True

    # Test 2: Posisi di luar map
    assert in_bounds((-1, 2)) is False

    # Test 3: Tembok tidak bisa dilewati
    assert is_walkable((2, 2)) is False

    # Test 4: Tambahkan tembok pada garis pandang
    GRID[1][3] = WALL

    assert can_enemy_see_player(
        ENEMY_POS, PLAYER_POS, SIGHT_RADIUS
    ) is False

    # Kembalikan kondisi grid
    GRID[1][3] = FLOOR

    # Test 5: Player terlalu jauh
    assert can_enemy_see_player(
        ENEMY_POS, PLAYER_POS, 2
    ) is False

    print("Semua tes dasar berhasil!")
